# Optimal policy evaluation using Temporal Differences Learning

Consider the MDP carachterized by $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$.

We consider the common case where the probabilities of transition are unknown, i.e. $\mathcal{P}:=\{P^{a}:\, a\in\mathcal{A}\}$ is unknown.

>A common approach to approximate the optimal policy consists in alternating the following two steps:
>1) updating the value function by the temporal differences rule TD(0);
>2) updating the policy by chosing the action that maximizes the value function.

**The TD(0) estimation of the value function**:

The value function is updated as follows:

$$
v_{\pi}(s_{n}) \leftarrow v_{\pi}(s_{n}) + \eta (R_{n+1} + \gamma * v_{\pi}(s_{n+1}) − v_{\pi}(s_{n})),
$$

where $\eta > 0$ is the increment.

**The TD(n) estimation of the value function**:

Under the assumptions of a stationary policy, consider the value function

$$
v_{\pi,n}(s):=\mathbb{E}_{\pi} \left[ G_{n} \mid S_{n}=s \right],
$$

where $G_{n}:=\sum\limits_{k=0}^{T \wedge \tau} \gamma^{k} R_{1+k+n} $ is the *objective function* at time $0$, with $R_{1+k+n}$ the reward at time $1+k+n$, $T$ a finite horizon and $\tau$ the minimum time to reach an absorbing state. The objective function selects a random trajectory starting from state $s$ under the policy $\pi$.

The value function is updated as follows:

$$
v_{\pi}(s_{n}) \leftarrow v_{\pi}(s_{n}) + \eta (G_{n} − v_{\pi}(s_{n})),
$$

where $\eta > 0$ is the increment.

In [28]:
import numpy as  np

In [ ]:
def TD0(P_trans_allow, Pi, reward, N_max, eta, v_0, s_0, gamma, absorbing_state=None):
    '''
    Estimates the value function v_pi using TD(0).

    P_trans_allow[a, s, s'] : 1 if the transition s -> s' is allowed under action a
                              (the true transition probabilities are unknown)
    Pi[s]         : action chosen by the deterministic policy in state s
    reward[a, s, s']   : reward matrix
    N_max         : number of updates
    eta           : learning rate (step size)
    v_0           : initial values of v
    s_0           : initial state
    gamma         : discount factor
    absorbing_state : list of absorbing states known by the expert
    '''

    if absorbing_state is None:          # pas d'etat absorbant fourni : liste vide
        absorbing_state = []

    v = np.array(v_0, dtype=float)
    n_states = reward.shape[0]

    non_absorbing = [s for s in range(n_states) if s not in absorbing_state]

    s = s_0

    for _ in range(N_max):
        # pi(a|s)=1
        a = Pi[s]

        # Nouvel etat : tirage uniforme parmi les etats autorises
        possible_next_states = np.flatnonzero(P_trans_allow[a, s] > 0)
        if possible_next_states.size == 0:
            raise ValueError(
                f"No allowed next state for state {s}, action {a}. "
                "Each row of P_trans_allow[a, s, :] must contain at least one non-zero entry."
            )
        
        s_prime = np.random.choice(possible_next_states)

        # Mise a jour TD(0)
        v[s] = v[s] + eta * (reward[a, s, s_prime] + gamma * v[s_prime] - v[s])

        # Si s_prime est absorbant : on repart d'un etat non absorbant choisi
        # uniformement ; sinon on continue depuis S_prime
        if s_prime in absorbing_state:
            s = np.random.choice(non_absorbing)
        else:
            s = s_prime

    return v

## Example:

Small example from the course.

In [44]:
N_max = 10000      # number of trajectories

# states = [high, low]
# actions = [s, w, c]

R_s = np.array([[5, 5], [-10, 5]])
R_w = np.array([[1, 0], [0, 1]])
R_c = np.array([[0, 0], [0, 0]])
R_complet = np.stack([R_s, R_w, R_c], axis=0) 
# matrix of rewards

P_s_allowed = np.array([[1, 1], [1, 1]])          
P_w_allowed = np.array([[1, 0], [0, 1]])
P_c_allowed = np.array([[1 ,0], [1 ,0]])
P_act_allowed = np.stack([P_s_allowed, P_w_allowed, P_c_allowed], axis=0)
# environmental matrix P[a,s,s'] with 1 where the transition is eventually allowed

pi_s_w = [0, 1]
pi_s_s = [0, 0]
pi_s_c = [0, 2]
# 3 deterministic policy on states

gamma = 0.8

eta_1 = 0.5
eta_2 = 0.05
eta_3 = 0.01

v_0 = [50,50]

s_0 = 0

print("Politique s/w, eta 1",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_1, v_0, s_0, gamma))
print("Politique s/s, eta 1",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_1, v_0, s_0, gamma))
print( "Politque s/c, eta 1",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_1, v_0, s_0, gamma))

print("Politique s/w, eta 2",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_2, v_0, s_0, gamma))
print("Politique s/s, eta 2",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_2, v_0, s_0, gamma))
print( "Politque s/c, eta 2",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_2, v_0, s_0, gamma))

print("Politique s/w, eta 3",TD0(P_act_allowed, pi_s_w, R_complet, N_max, eta_3, v_0, s_0, gamma))
print("Politique s/s, eta 3",TD0(P_act_allowed, pi_s_s, R_complet, N_max, eta_3, v_0, s_0, gamma))
print( "Politque s/c, eta 3",TD0(P_act_allowed, pi_s_c, R_complet, N_max, eta_3, v_0, s_0, gamma))

Politique s/w, eta 1 [46.25  5.  ]
Politique s/s, eta 1 [ 4.37579675 -4.04615569]
Politque s/c, eta 1 [17.55578356 13.60748917]
Politique s/w, eta 2 [49.75  5.  ]
Politique s/s, eta 2 [8.82259152 1.28781029]
Politque s/c, eta 2 [18.18459601 14.19244579]
Politique s/w, eta 3 [49.851099    5.00000009]
Politique s/s, eta 3 [9.84649012 2.36287668]
Politque s/c, eta 3 [17.64387633 14.21040107]


## Example:

Small example from the course.

In [58]:
# allowed trajectories
P_act_allow_s = np.array([
    [1, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]])

P_act_allow_w = np.array([
    [1, 0, 1, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1],   
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]])

P_act_allowed_etudiant = np.stack([P_act_allow_s, P_act_allow_w], axis=0)

# rewards
R_allow_s = np.array([
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, -10, 0, 0],
    [0, 1, -1, 0, 0, 0, 0],
    [0, 0, 0, -10, 0, 100, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])

R_allow_w = np.array([
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0],
    [0, 0, -1, -10, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, -1000],   
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]])
R_allowed_etudiant = np.stack([R_allow_s, R_allow_w], axis=0)

absorbing_state = [4, 5, 6]
pi_SWWS = [0, 1, 1, 0, 0, 0, 0]   # Repose, Travaille, Travaille, Repose
v_0_etudiant = [0] * 7
s_0 = 3
N_max = 100000
gamma = 0.8


print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.01, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.05, v_0_etudiant, s_0, gamma, absorbing_state))
print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.1, v_0_etudiant, s_0, gamma, absorbing_state))

[15.27570014 20.34652461 40.67432481 75.77703227  0.          0.
  0.        ]
[13.09405182 18.68469626 41.56797731 77.59311959  0.          0.
  0.        ]
[15.57042463 21.51196101 45.93672842 80.50370202  0.          0.
  0.        ]


In [63]:
print(TD0(P_act_allowed_etudiant, pi_SWWS, R_allowed_etudiant, N_max, 0.5, v_0_etudiant, s_0, gamma, absorbing_state))

[19.47072125 33.77361497 61.30871262 97.58881732  0.          0.
  0.        ]
